# ROGII | LGBM Aug v2 â€” Enhanced Features + Cal-Zone Augmentation

**New in v2 (from super-solution analysis):**
- `FormationPlaneKNN`: all 6 formations (ANCC, ASTNU, ASTNL, EGFDU, EGFDL, BUDA)
- `wls_b_well`: exponential-decay weighted calibration
- Multi-scale self-correlation (hw=8,15,25): 3 NCC TVT signals
- 7 beam configs with Â±2 delta steps
- Formation consensus features (std/range across 6 formations)
- Inter-signal consensus (`signal_std`)
- GR envelope + energy features
- Affine GR calibration (scale a + offset b)
- Per-formation known-zone RMSE

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.interpolate import interp1d
from scipy.spatial import cKDTree
from lightgbm import LGBMRegressor
from joblib import Parallel, delayed
import multiprocessing
import glob
import time
import gc
import os

if os.path.exists('/kaggle/input'):
    c1 = Path('/kaggle/input/rogii-wellbore-geology-prediction')
    c2 = Path('/kaggle/input/competitions/rogii-wellbore-geology-prediction')
    DATA_DIR = c1 if (c1 / 'test').exists() else (c2 if (c2 / 'test').exists() else c1)
    OUT_DIR = Path('/kaggle/working')
else:
    DATA_DIR = Path('../../data').resolve()
    OUT_DIR = Path('.')

TRAIN_DIR = DATA_DIR / 'train'
TEST_DIR = DATA_DIR / 'test'
print(f'DATA_DIR: {DATA_DIR}')
print(f'Train wells: {len(list(TRAIN_DIR.glob("*__horizontal_well.csv")))}')
print(f'Test wells:  {len(list(TEST_DIR.glob("*__horizontal_well.csv")))}')

In [ ]:
# â”€â”€ Global Config â”€â”€
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DEMO = False             # set True for quick validation run
ONLINE_TRAINING = True
N_AUG_SPLITS    = 1 if DEMO else 3
MIN_KNOWN_FOR_AUG = 20
N_JOBS = min(4, multiprocessing.cpu_count())

FORMATIONS = ["ANCC", "ASTNU", "ASTNL", "EGFDU", "EGFDL", "BUDA"]

BEAMS = [
    (10, 20.0, 144.0, 3, "cons"),
    (10,  8.0,  64.0, 3, "loose"),
    ( 8, 35.0, 220.0, 1, "vcons"),
    (10, 14.0,  90.0, 5, "sm5"),
    (20,  4.0,  36.0, 3, "vloose"),
    (12, 12.0, 100.0, 3, "mid"),
    (15, 25.0, 180.0, 2, "stiff"),
]

LGBM_PARAMS = dict(
    n_estimators=100 if DEMO else 3000,
    learning_rate=0.03,
    num_leaves=31 if DEMO else 64,
    min_child_samples=50,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=5.0,
    reg_alpha=0.1,
    objective='regression',
    random_state=RANDOM_STATE,
    verbose=-1,
    n_jobs=-1,
)

PF_N_PARTICLES = 500
PF_MOMENTUM_ALPHA = 0.993
PF_Z_SIGMA_FLOOR = 0.005
PF_Z_SIGMA_SCALE = 2.0
PF_VELOCITY_NOISE_STD = 0.005
PF_POSITION_NOISE_STD = 0.01
PF_INIT_VELOCITY_STD = 0.02
PF_GR_SIGMA_MIN = 10.0
PF_GR_SIGMA_MAX = 60.0
PF_GR_SIGMA_DEFAULT = 30.0
PF_INIT_SPREAD_STD = 0.5
PF_RESAMPLE_THRESHOLD = 0.5
PF_ROUGHENING_STD_POS = 0.2
PF_ROUGHENING_STD_VEL = 0.003
PF_GR_ROLLING_WINDOW = 5
PF_GR_ROLLING_WEIGHT = 0.3

ANCC_ALPHA = 0.998
ANCC_RATE_NOISE_STD = 0.002
ANCC_POS_NOISE_STD = 0.005
ANCC_INIT_RATE_STD = 0.01
ANCC_INIT_SPREAD_STD = 0.3
ANCC_ROUGHENING_STD_POS = 0.1
ANCC_ROUGHENING_STD_RATE = 0.001
ANCC_N_PARTICLES = 500

SPATIAL_PLANE_K = 10
DENSE_SAMPLES_PER_WELL = 40
DENSE_K = 20

print(f'DEMO={DEMO}, N_JOBS={N_JOBS}, N_AUG_SPLITS={N_AUG_SPLITS}, ONLINE_TRAINING={ONLINE_TRAINING}')
print(f'FORMATIONS={FORMATIONS}')
print(f'BEAMS={len(BEAMS)} configs')

In [ ]:
# â”€â”€ Helpers â”€â”€
def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2)))

def safe_interp(x, xp, fp):
    xp = np.asarray(xp, dtype=np.float64)
    fp = np.asarray(fp, dtype=np.float64)
    mask = np.isfinite(xp) & np.isfinite(fp)
    if mask.sum() < 2:
        return np.full(len(np.asarray(x)), np.nan)
    order = np.argsort(xp[mask])
    return np.interp(np.asarray(x, float), xp[mask][order], fp[mask][order],
                     left=np.nan, right=np.nan)

def robust_slope(x, y, default=0.0):
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 2:
        return default
    x, y = x[mask], y[mask]
    if np.std(x) < 1e-6:
        return default
    return float(np.polyfit(x, y, 1)[0])

def affine_cal(kgr, tw_at_k, min_pts=20):
    v = np.isfinite(kgr) & np.isfinite(tw_at_k)
    if v.sum() < min_pts or np.std(tw_at_k[v]) < 1e-6:
        bias = float(np.nanmean(kgr) - np.nanmean(tw_at_k)) if v.any() else 0.0
        return 1.0, bias
    a, b = np.polyfit(tw_at_k[v], kgr[v], 1)
    return float(a), float(b)

def wls_b_well(ktvt, kz, form_kn_col, decay=0.02):
    n = len(ktvt)
    if n < 3:
        return float(np.median(ktvt + kz - form_kn_col))
    w = np.exp(decay * np.arange(n)); w /= w.sum()
    return float(np.dot(w, ktvt + kz - form_kn_col))

def multi_scale_sc(kgr, ktvt, hgr, hws=(8, 15, 25), stride=3):
    results = []
    fallback = float(ktvt[-1]) if len(ktvt) > 0 else 0.0
    nh = len(hgr)
    for hw_sc in hws:
        win = 2 * hw_sc + 1
        nk = len(kgr)
        if nk < win + 1 or nh == 0:
            results.append((np.full(nh, fallback, np.float32), np.zeros(nh, np.float32)))
            continue
        kg = pd.Series(kgr).rolling(5, center=True, min_periods=1).mean().values.astype(np.float32)
        hg = pd.Series(hgr).rolling(5, center=True, min_periods=1).mean().values.astype(np.float32)
        sts = np.arange(0, nk - win + 1, stride, dtype=np.int32)
        if len(sts) == 0:
            results.append((np.full(nh, fallback, np.float32), np.zeros(nh, np.float32)))
            continue
        C = kg[sts[:, None] + np.arange(win, dtype=np.int32)[None, :]].astype(np.float32)
        Cn = (C - C.mean(1, keepdims=True)) / (C.std(1, keepdims=True) + 1e-6)
        hp = np.pad(hg, hw_sc, mode='edge')
        H = hp[np.arange(nh)[:, None] + np.arange(win)[None, :]].astype(np.float32)
        Hn = (H - H.mean(1, keepdims=True)) / (H.std(1, keepdims=True) + 1e-6)
        ncc = Hn @ Cn.T / win
        best = ncc.argmax(1)
        score = ncc.max(1).astype(np.float32)
        ctrs = np.clip(sts[best] + hw_sc, 0, nk - 1)
        results.append((ktvt[ctrs].astype(np.float32), score))
    return results

def gr_envelope(gr, w=21):
    return pd.Series(gr).rolling(w, center=True, min_periods=1).max().values.astype(np.float32)

def gr_energy(gr, w=21):
    return np.sqrt(pd.Series(gr.astype(float) ** 2).rolling(
        w, center=True, min_periods=1).mean().values.clip(0)).astype(np.float32)

In [ ]:
# â”€â”€ Beam Search â”€â”€
def nearest_idx(sorted_arr, value):
    idx = int(np.searchsorted(sorted_arr, value, side='left'))
    if idx >= len(sorted_arr):
        return len(sorted_arr) - 1
    if idx > 0 and abs(sorted_arr[idx - 1] - value) <= abs(sorted_arr[idx] - value):
        return idx - 1
    return idx

def smooth_gr(values, fallback, radius):
    s = pd.Series(values, dtype='float32').interpolate(limit_direction='both').fillna(fallback)
    if radius > 0:
        s = s.rolling(radius * 2 + 1, center=True, min_periods=1).mean()
    return s.to_numpy(dtype=np.float32)

def beam_search(gr_hidden, tw_tvt, tw_gr, start_tvt,
                beam_size=10, move_cost=20.0, emit_scale=144.0, radius=2):
    tw_tvt = np.asarray(tw_tvt, dtype=np.float32)
    tw_gr = np.asarray(tw_gr, dtype=np.float32)
    T = len(tw_tvt)
    fallback = float(np.nanmean(tw_gr))
    gr_smooth = smooth_gr(gr_hidden, fallback, radius)
    start = nearest_idx(tw_tvt, start_tvt)
    beam_idx = np.full(beam_size, start, dtype=np.int32)
    beam_cost = np.zeros(beam_size, dtype=np.float64)
    n_steps = len(gr_smooth)
    bp_state = np.empty((n_steps, beam_size), dtype=np.int32)
    bp_idx = np.empty((n_steps, beam_size), dtype=np.int32)
    deltas = np.array([-2, -1, 0, 1, 2], dtype=np.int32)   # +-2 delta steps
    for step, gr_val in enumerate(gr_smooth):
        cand_idx = np.clip(beam_idx[:, None] + deltas[None, :], 0, T - 1)
        emit = (gr_val - tw_gr[cand_idx]) ** 2 / emit_scale
        move = move_cost * np.abs(deltas)[None, :]
        cand_cost = beam_cost[:, None] + emit + move
        flat_idx = cand_idx.ravel()
        flat_cost = cand_cost.ravel()
        flat_par = np.repeat(np.arange(beam_size), len(deltas))
        order = np.argsort(flat_cost, kind='stable')
        kept, seen = [], set()
        for o in order:
            ti = int(flat_idx[o])
            if ti not in seen:
                seen.add(ti)
                kept.append(o)
                if len(kept) == beam_size:
                    break
        while len(kept) < beam_size:
            kept.append(kept[-1])
        kept = np.array(kept, dtype=np.int32)
        bp_state[step] = flat_par[kept]
        bp_idx[step] = flat_idx[kept]
        beam_idx = flat_idx[kept].astype(np.int32)
        beam_cost = flat_cost[kept]
    path = np.empty(n_steps, dtype=np.int32)
    cur_b = int(np.argmin(beam_cost))
    for step in range(n_steps - 1, -1, -1):
        path[step] = bp_idx[step, cur_b]
        cur_b = bp_state[step, cur_b]
    return tw_tvt[path]

def run_beam(hw, tw, beam_size=10, move_cost=20.0, emit_scale=144.0, radius=3):
    tw_tvt = tw['TVT'].values.astype(np.float32)
    tw_gr  = tw['GR'].values.astype(np.float32)
    known  = hw[hw['TVT_input'].notna()]
    evalz  = hw[hw['TVT_input'].isna()]
    if len(evalz) == 0 or len(known) == 0:
        return np.array([]), np.array([])
    last_tvt = float(known['TVT_input'].iloc[-1])
    gr_filled = hw['GR'].astype(float).interpolate(limit_direction='both')
    gr_filled = gr_filled.fillna(float(np.nanmean(tw_gr)))
    hidden_gr = gr_filled.iloc[evalz.index[0]:].values.astype(np.float32)
    pred = beam_search(hidden_gr, tw_tvt, tw_gr, last_tvt,
                       beam_size=beam_size, move_cost=move_cost,
                       emit_scale=emit_scale, radius=radius)
    n_eval = len(evalz)
    pred = pred[:n_eval]
    return pred, np.zeros(n_eval, dtype=np.float32)

def run_all_beams(hw, tw):
    tw_tvt = tw['TVT'].values.astype(np.float32)
    tw_gr  = tw['GR'].values.astype(np.float32)
    known  = hw[hw['TVT_input'].notna()]
    evalz  = hw[hw['TVT_input'].isna()]
    if len(evalz) == 0 or len(known) == 0:
        return {tag: None for _, _, _, _, tag in BEAMS}
    last_tvt = float(known['TVT_input'].iloc[-1])
    gr_filled = hw['GR'].astype(float).interpolate(limit_direction='both')
    gr_filled = gr_filled.fillna(float(np.nanmean(tw_gr)))
    hidden_gr = gr_filled.iloc[evalz.index[0]:].values.astype(np.float32)
    n_eval = len(evalz)
    result = {}
    for bs, mc, es, r, tag in BEAMS:
        pred = beam_search(hidden_gr, tw_tvt, tw_gr, last_tvt,
                           beam_size=bs, move_cost=mc, emit_scale=es, radius=r)
        result[tag] = pred[:n_eval].astype(np.float32)
    return result

In [ ]:
def pf_calibrate_gr_sigma(hw, tw_tvt, tw_gr):
    known = hw[hw['TVT_input'].notna()]
    known_gr = known[known['GR'].notna()]
    if len(known_gr) < 20:
        return PF_GR_SIGMA_DEFAULT
    tw_func = interp1d(tw_tvt, tw_gr, bounds_error=False, fill_value=(tw_gr[0], tw_gr[-1]))
    expected = tw_func(known_gr['TVT_input'].values)
    residuals = known_gr['GR'].values - expected
    return np.clip(np.std(residuals), PF_GR_SIGMA_MIN, PF_GR_SIGMA_MAX)

def pf_estimate_init_velocity(hw):
    known = hw[hw['TVT_input'].notna()]
    if len(known) < 10:
        return 0.0
    tail = known.tail(20)
    if len(tail) < 5:
        return 0.0
    dtvt = np.diff(tail['TVT_input'].values)
    dmd = np.diff(tail['MD'].values)
    mask = dmd > 0
    if mask.sum() < 3:
        return 0.0
    return np.median(dtvt[mask] / dmd[mask])

def pf_learn_z_beta(hw):
    known = hw[hw['TVT_input'].notna()]
    if len(known) < 30:
        return -1.0, 0.0, 0.1
    dz = np.diff(known['Z'].values)
    dtvt = np.diff(known['TVT_input'].values)
    dmd = np.diff(known['MD'].values)
    mask = dmd > 0
    if mask.sum() < 10:
        return -1.0, 0.0, 0.1
    vz = dz[mask] / dmd[mask]
    vt = dtvt[mask] / dmd[mask]
    A = np.column_stack([vz, np.ones_like(vz)])
    coef, _, _, _ = np.linalg.lstsq(A, vt, rcond=None)
    residuals = vt - (coef[0] * vz + coef[1])
    sigma = max(np.std(residuals), 0.001)
    return coef[0], coef[1], sigma

def run_pf_z_velocity(hw, tw_tvt, tw_gr, n_particles=PF_N_PARTICLES):
    tw_func_point = interp1d(tw_tvt, tw_gr, bounds_error=False, fill_value=(tw_gr[0], tw_gr[-1]))
    tw_smooth_gr = pd.Series(tw_gr).rolling(PF_GR_ROLLING_WINDOW, center=True, min_periods=1).mean().values
    tw_func_smooth = interp1d(tw_tvt, tw_smooth_gr, bounds_error=False, fill_value=(tw_smooth_gr[0], tw_smooth_gr[-1]))
    tvt_min, tvt_max = tw_tvt.min(), tw_tvt.max()
    gr_sigma = pf_calibrate_gr_sigma(hw, tw_tvt, tw_gr)
    beta, intercept, z_sigma = pf_learn_z_beta(hw)
    known = hw[hw['TVT_input'].notna()]
    evalz = hw[hw['TVT_input'].isna()]
    if len(evalz) == 0:
        return np.array([]), np.array([])
    hw_gr_smooth = hw['GR'].rolling(PF_GR_ROLLING_WINDOW, center=True, min_periods=1).mean()
    last_tvt = known['TVT_input'].iloc[-1]
    positions = last_tvt + np.random.normal(0, PF_INIT_SPREAD_STD, n_particles)
    init_v = pf_estimate_init_velocity(hw)
    velocities = init_v + np.random.normal(0, PF_INIT_VELOCITY_STD, n_particles)
    weights = np.ones(n_particles) / n_particles
    eval_indices = evalz.index.tolist()
    md_vals = evalz['MD'].values
    gr_vals = evalz['GR'].values
    z_vals = evalz['Z'].values
    prev_md = known['MD'].iloc[-1]
    prev_z = known['Z'].iloc[-1]
    pred_tvts = np.empty(len(evalz))
    pred_stds = np.empty(len(evalz))
    for i, idx in enumerate(eval_indices):
        d_md = md_vals[i] - prev_md
        if d_md <= 0:
            d_md = 1.0
        dz_dmd = (z_vals[i] - prev_z) / d_md
        v_expected = beta * dz_dmd + intercept
        velocities = (PF_MOMENTUM_ALPHA * velocities
                      + np.random.normal(0, PF_VELOCITY_NOISE_STD, n_particles))
        positions = (positions + velocities * d_md
                     + np.random.normal(0, PF_POSITION_NOISE_STD, n_particles))
        positions = np.clip(positions, tvt_min - 50, tvt_max + 50)
        if not np.isnan(gr_vals[i]):
            gr_smooth = hw_gr_smooth.iloc[hw.index.get_loc(idx)]
            expected_point = tw_func_point(positions)
            diff_point = gr_vals[i] - expected_point
            lik_point = np.exp(-0.5 * (diff_point / gr_sigma) ** 2)
            if not np.isnan(gr_smooth):
                expected_smooth = tw_func_smooth(positions)
                diff_smooth = gr_smooth - expected_smooth
                lik_smooth = np.exp(-0.5 * (diff_smooth / (gr_sigma * 1.5)) ** 2)
                likelihood = ((1 - PF_GR_ROLLING_WEIGHT) * lik_point
                              + PF_GR_ROLLING_WEIGHT * lik_smooth)
            else:
                likelihood = lik_point
            likelihood = np.maximum(likelihood, 1e-300)
            weights = weights * likelihood
            w_sum = weights.sum()
            if w_sum > 0:
                weights /= w_sum
            else:
                weights[:] = 1.0 / n_particles
        z_sig = max(z_sigma * PF_Z_SIGMA_SCALE, PF_Z_SIGMA_FLOOR)
        diff_v = velocities - v_expected
        lik_z = np.exp(-0.5 * (diff_v / z_sig) ** 2)
        lik_z = np.maximum(lik_z, 1e-300)
        weights = weights * lik_z
        w_sum = weights.sum()
        if w_sum > 0:
            weights /= w_sum
        else:
            weights[:] = 1.0 / n_particles
        n_eff = 1.0 / np.sum(weights ** 2)
        if n_eff < PF_RESAMPLE_THRESHOLD * n_particles:
            cum = np.cumsum(weights)
            pos_resample = (np.arange(n_particles) + np.random.uniform()) / n_particles
            indices = np.searchsorted(cum, pos_resample)
            positions = positions[indices]
            velocities = velocities[indices]
            weights[:] = 1.0 / n_particles
            positions += np.random.normal(0, PF_ROUGHENING_STD_POS, n_particles)
            velocities += np.random.normal(0, PF_ROUGHENING_STD_VEL, n_particles)
        pred_tvts[i] = np.average(positions, weights=weights)
        pred_stds[i] = np.sqrt(np.average((positions - pred_tvts[i]) ** 2, weights=weights))
        prev_md = md_vals[i]
        prev_z = z_vals[i]
    return pred_tvts, pred_stds

In [ ]:
def ancc_estimate_init_rate(hw):
    known = hw[hw['TVT_input'].notna()]
    if len(known) < 10:
        return 0.0
    tail = known.tail(30)
    dtvt = np.diff(tail['TVT_input'].values)
    dz = np.diff(tail['Z'].values)
    dmd = np.diff(tail['MD'].values)
    dancc = dtvt + dz
    mask = dmd > 0
    if mask.sum() < 3:
        return 0.0
    return np.median(dancc[mask] / dmd[mask])

def run_pf_ancc(hw, tw_tvt, tw_gr, n_particles=ANCC_N_PARTICLES):
    tvt_min, tvt_max = tw_tvt.min(), tw_tvt.max()
    gr_sigma = pf_calibrate_gr_sigma(hw, tw_tvt, tw_gr)
    known = hw[hw['TVT_input'].notna()]
    evalz = hw[hw['TVT_input'].isna()]
    if len(evalz) == 0:
        return np.array([]), np.array([])
    last_state = known['TVT_input'].iloc[-1] + known['Z'].iloc[-1]
    init_rate = ancc_estimate_init_rate(hw)
    pos = last_state + np.random.normal(0, ANCC_INIT_SPREAD_STD, n_particles)
    rate = init_rate + np.random.normal(0, ANCC_INIT_RATE_STD, n_particles)
    w = np.ones(n_particles) / n_particles
    md_vals = evalz['MD'].values
    z_vals = evalz['Z'].values
    gr_vals = evalz['GR'].values
    prev_md = known['MD'].iloc[-1]
    pred_tvts = np.empty(len(evalz))
    pred_stds = np.empty(len(evalz))
    for i in range(len(evalz)):
        d_md = md_vals[i] - prev_md
        if d_md <= 0:
            d_md = 1.0
        rate = ANCC_ALPHA * rate + np.random.normal(0, ANCC_RATE_NOISE_STD, n_particles)
        pos = pos + rate * d_md + np.random.normal(0, ANCC_POS_NOISE_STD, n_particles)
        tvt_est = pos - z_vals[i]
        tvt_clipped = np.clip(tvt_est, tvt_min - 50, tvt_max + 50)
        pos = tvt_clipped + z_vals[i]
        if not np.isnan(gr_vals[i]):
            expected_gr = np.interp(tvt_clipped, tw_tvt, tw_gr)
            diff = gr_vals[i] - expected_gr
            lik = np.exp(-0.5 * (diff / gr_sigma) ** 2)
            lik = np.maximum(lik, 1e-300)
            w *= lik
            w_sum = w.sum()
            if w_sum > 0:
                w /= w_sum
            else:
                w[:] = 1.0 / n_particles
        n_eff = 1.0 / np.sum(w ** 2)
        if n_eff < PF_RESAMPLE_THRESHOLD * n_particles:
            cum = np.cumsum(w)
            u = (np.arange(n_particles) + np.random.uniform()) / n_particles
            idx = np.searchsorted(cum, u)
            pos = pos[idx]
            rate = rate[idx]
            w[:] = 1.0 / n_particles
            pos += np.random.normal(0, ANCC_ROUGHENING_STD_POS, n_particles)
            rate += np.random.normal(0, ANCC_ROUGHENING_STD_RATE, n_particles)
        tvt_weighted = np.average(pos - z_vals[i], weights=w)
        pred_tvts[i] = tvt_weighted
        pred_stds[i] = np.sqrt(np.average((pos - z_vals[i] - tvt_weighted) ** 2, weights=w))
        prev_md = md_vals[i]
    return pred_tvts, pred_stds

In [ ]:
# â”€â”€ Spatial Formation Imputers â”€â”€
class FormationPlaneKNN:
    def __init__(self, well_ids, data_dir):
        rows = []
        for wid in well_ids:
            p = data_dir / f'{wid}__horizontal_well.csv'
            try:
                df = pd.read_csv(p, usecols=['X', 'Y'] + FORMATIONS).dropna()
            except Exception:
                continue
            if len(df) == 0:
                continue
            row = {'wid': wid, 'x': float(df['X'].median()), 'y': float(df['Y'].median())}
            for c in FORMATIONS:
                row[f'{c}_m'] = float(df[c].median())
            rows.append(row)
        self.df = pd.DataFrame(rows)
        self.wmap = {w: i for i, w in enumerate(self.df['wid'])}
        xy = self.df[['x', 'y']].to_numpy()
        self.scale = np.where(xy.std(0) < 1e-3, 1.0, xy.std(0))
        self.tree = cKDTree(xy / self.scale)
        self.xa = self.df['x'].to_numpy()
        self.ya = self.df['y'].to_numpy()
        self.fa = self.df[[f'{c}_m' for c in FORMATIONS]].to_numpy(np.float64)  # (n_wells, 6)

    def impute(self, xy_query, self_wid=None, k=SPATIAL_PLANE_K):
        xy_query = np.atleast_2d(np.asarray(xy_query, dtype=np.float64))
        q = xy_query / self.scale
        n_fetch = min(k + 5, len(self.df))
        dist, idx = self.tree.query(q, k=n_fetch)
        if self_wid is not None and self_wid in self.wmap:
            dist = np.where(idx == self.wmap[self_wid], np.inf, dist)
        order = np.argpartition(dist, kth=min(k - 1, n_fetch - 1), axis=1)[:, :k]
        dk = np.take_along_axis(dist, order, axis=1)
        ik = np.take_along_axis(idx, order, axis=1)
        valid = np.isfinite(dk)
        w = np.where(valid, 1.0 / (dk + 1e-3), 0.0).astype(np.float64)
        xn = self.xa[ik]; yn = self.ya[ik]; fn = self.fa[ik]  # (n_q, k, 6)
        wx = w * xn; wy = w * yn
        ATA = np.zeros((len(xy_query), 3, 3))
        ATA[:, 0, 0] = (wx * xn).sum(1); ATA[:, 0, 1] = (wx * yn).sum(1); ATA[:, 0, 2] = wx.sum(1)
        ATA[:, 1, 0] = ATA[:, 0, 1]; ATA[:, 1, 1] = (wy * yn).sum(1); ATA[:, 1, 2] = wy.sum(1)
        ATA[:, 2, 0] = ATA[:, 0, 2]; ATA[:, 2, 1] = ATA[:, 1, 2]; ATA[:, 2, 2] = w.sum(1)
        ATA[:, 0, 0] += 1e-9; ATA[:, 1, 1] += 1e-9; ATA[:, 2, 2] += 1e-9
        rhs = np.stack([(wx[:, :, None] * fn).sum(1),
                        (wy[:, :, None] * fn).sum(1),
                        (w[:, :, None] * fn).sum(1)], axis=1)  # (n_q, 3, 6)
        try:
            coef = np.linalg.solve(ATA, rhs)   # (n_q, 3, 6)
        except np.linalg.LinAlgError:
            coef = np.zeros((len(xy_query), 3, 6))
            for ri in range(len(xy_query)):
                try:
                    coef[ri] = np.linalg.pinv(ATA[ri]) @ rhs[ri]
                except Exception:
                    pass
        Xq = xy_query[:, 0]; Yq = xy_query[:, 1]
        pred = (Xq[:, None] * coef[:, 0, :] +
                Yq[:, None] * coef[:, 1, :] +
                coef[:, 2, :]).astype(np.float32)  # (n_q, 6)
        no_valid = ~valid.any(1)
        pred[no_valid] = self.fa.mean(0).astype(np.float32)
        min_dist = np.where(valid, dk, np.inf).min(1).astype(np.float32)
        return pred, min_dist   # (n_q, 6), (n_q,)


class DenseANCCImputer:
    def __init__(self, well_ids, data_dir, samples_per_well=DENSE_SAMPLES_PER_WELL):
        xs, ys, anccs, wids = [], [], [], []
        for wid in well_ids:
            p = data_dir / f'{wid}__horizontal_well.csv'
            try:
                df = pd.read_csv(p, usecols=['X', 'Y', 'ANCC']).dropna()
            except Exception:
                continue
            if len(df) == 0:
                continue
            idx = np.linspace(0, len(df) - 1, min(samples_per_well, len(df)), dtype=int)
            sampled = df.iloc[idx]
            xs.append(sampled['X'].values)
            ys.append(sampled['Y'].values)
            anccs.append(sampled['ANCC'].values)
            wids.extend([wid] * len(sampled))
        self.xy = np.column_stack([np.concatenate(xs), np.concatenate(ys)])
        self.ancc = np.concatenate(anccs).astype(np.float32)
        self.wids = np.array(wids)
        self.wid_set = set(wids)
        self.scale = self.xy.std(axis=0)
        self.scale = np.where(self.scale < 1e-3, 1.0, self.scale)
        self.tree = cKDTree(self.xy / self.scale)

    def impute(self, xy_query, self_wid=None, k=DENSE_K):
        xy_query = np.atleast_2d(xy_query)
        q = xy_query / self.scale
        n_fetch = min(k + (DENSE_SAMPLES_PER_WELL + 5), len(self.ancc))
        dist, idx = self.tree.query(q, k=n_fetch, workers=-1)
        if self_wid is not None and self_wid in self.wid_set:
            mask_self = self.wids[idx] == self_wid
            dist = np.where(mask_self, np.inf, dist)
        order = np.argpartition(dist, kth=min(k - 1, n_fetch - 1), axis=1)[:, :k]
        dk = np.take_along_axis(dist, order, axis=1)
        ik = np.take_along_axis(idx, order, axis=1)
        valid = np.isfinite(dk)
        w = np.where(valid, 1.0 / (dk + 1e-3), 0.0)
        sw = w.sum(axis=1); no_n = sw < 1e-9; safe = np.where(no_n, 1.0, sw)
        ancc_n = self.ancc[ik]
        ancc_pred = (ancc_n * w).sum(axis=1) / safe
        ancc_pred = np.where(no_n, float(self.ancc.mean()), ancc_pred)
        diff_sq = (ancc_n - ancc_pred[:, None]) ** 2
        var = (diff_sq * w).sum(axis=1) / safe
        neighbor_std = np.sqrt(np.maximum(var, 0.0))
        min_dist = np.where(valid, dk, np.inf).min(axis=1)
        return (ancc_pred.astype(np.float32),
                neighbor_std.astype(np.float32),
                min_dist.astype(np.float32))

In [ ]:
# â”€â”€ Feature Engineering â”€â”€
def build_features(hw, tw, pf_pred, pf_std, form_imputer, dense_imputer, wid,
                   beams_dict=None, sc_results=None, is_train=True):
    known = hw[hw['TVT_input'].notna()]
    evalz = hw[hw['TVT_input'].isna()]
    if len(evalz) == 0 or len(known) == 0:
        return pd.DataFrame()

    last_known = known.iloc[-1]
    last_known_tvt = float(last_known['TVT_input'])
    ps_md = float(last_known['MD']); ps_z = float(last_known['Z'])
    ps_x = float(last_known['X']); ps_y = float(last_known['Y'])

    tw_tvt = tw['TVT'].values.astype(np.float64)
    tw_gr  = tw['GR'].values.astype(np.float64)
    n_eval = len(evalz)

    xy_eval  = evalz[['X', 'Y']].to_numpy()
    xy_known = known[['X', 'Y']].to_numpy()
    self_arg = wid if is_train else None
    z_eval   = evalz['Z'].values.astype(np.float32)
    z_known  = known['Z'].values.astype(np.float32)
    ktvt     = known['TVT_input'].values.astype(np.float32)

    # â”€â”€ FormationPlaneKNN: all 6 formations â”€â”€
    form_ev, knn_d = form_imputer.impute(xy_eval, self_wid=self_arg)   # (n_eval, 6)
    form_kn, _     = form_imputer.impute(xy_known, self_wid=self_arg)  # (n_known, 6)

    tvt_forms = {}; form_rmse_vals = {}; form_consistency = []
    for fi, fn in enumerate(FORMATIONS):
        b_all = float(np.median(ktvt + z_known - form_kn[:, fi]))
        b_wls = wls_b_well(ktvt, z_known, form_kn[:, fi])
        b_50  = float(np.median((ktvt + z_known - form_kn[:, fi])[-50:])) if len(ktvt) >= 5 else b_all
        tvt_f  = (-z_eval + form_ev[:, fi] + b_all).astype(np.float32)
        tvt_fw = (-z_eval + form_ev[:, fi] + b_wls).astype(np.float32)
        tvt_forms[fn]          = tvt_f
        tvt_forms[fn + '_wls'] = tvt_fw
        tvt_forms[f'b_{fn}']   = np.float32(b_all)
        tvt_forms[f'bw_{fn}']  = np.float32(b_wls)
        tvt_forms[f'b50_{fn}'] = np.float32(b_50)
        tvt_kn_pred = -z_known + form_kn[:, fi] + b_all
        form_rmse_vals[fn] = float(np.sqrt(np.mean((ktvt - tvt_kn_pred) ** 2)))
        form_consistency.append(tvt_f)

    spatial_tvt  = tvt_forms['ANCC']
    spatial_ancc = form_ev[:, 0].astype(np.float32)
    c_well = float(tvt_forms['b_ANCC'])

    form_stack   = np.stack(form_consistency, axis=1)  # (n_eval, 6)
    form_mean_d  = (form_stack.mean(1) - last_known_tvt).astype(np.float32)
    form_std_d   = form_stack.std(1).astype(np.float32)
    form_range_d = (form_stack.max(1) - form_stack.min(1)).astype(np.float32)

    # â”€â”€ Dense ANCC â”€â”€
    dense_ancc_eval, dense_nb_std_eval, dense_dist = dense_imputer.impute(xy_eval, self_wid=self_arg)
    dense_ancc_known, dense_nb_std_known, _        = dense_imputer.impute(xy_known, self_wid=self_arg)
    b_d_all  = float(np.median(ktvt + z_known - dense_ancc_known))
    b_d_wls  = wls_b_well(ktvt, z_known, dense_ancc_known)
    b_d_50   = float(np.median((ktvt + z_known - dense_ancc_known)[-50:])) if len(ktvt) >= 5 else b_d_all
    dense_tvt    = (-z_eval + dense_ancc_eval + b_d_all).astype(np.float32)
    dense_tvt_w  = (-z_eval + dense_ancc_eval + b_d_wls).astype(np.float32)
    dense_tvt_50 = (-z_eval + dense_ancc_eval + b_d_50).astype(np.float32)
    res_kn = ktvt + z_known - dense_ancc_known
    dense_known_rmse    = np.float32(np.sqrt(np.mean(res_kn ** 2)))
    dense_known_bias    = np.float32(np.mean(res_kn))
    dense_known_max_err = np.float32(np.max(np.abs(res_kn)))
    dense_known_nb_std_mean = np.float32(np.mean(dense_nb_std_known))

    # â”€â”€ Beam features â”€â”€
    beam_preds = {}
    for _, _, _, _, tag in BEAMS:
        if beams_dict is not None and tag in beams_dict and beams_dict[tag] is not None:
            arr = beams_dict[tag]
            beam_preds[tag] = (arr[:n_eval].astype(np.float32) if len(arr) >= n_eval
                               else np.full(n_eval, np.float32(last_known_tvt)))
        else:
            beam_preds[tag] = np.full(n_eval, np.float32(np.nan))

    beam_tags = [tag for _, _, _, _, tag in BEAMS]
    beam_stack = np.stack([beam_preds[t] for t in beam_tags], axis=1)  # (n_eval, 7)
    has_beams = ~np.isnan(beam_stack).any(axis=0)
    if has_beams.any():
        bv = beam_stack[:, has_beams]
        beam_mean_d = (bv.mean(1) - last_known_tvt).astype(np.float32)
        beam_std_d  = bv.std(1).astype(np.float32)
        beam_med_d  = (np.median(bv, axis=1) - last_known_tvt).astype(np.float32)
    else:
        beam_mean_d = beam_std_d = beam_med_d = np.full(n_eval, np.float32(np.nan))

    # â”€â”€ Multi-scale self-correlation â”€â”€
    if sc_results is not None and len(sc_results) == 3:
        sc8, sc8s   = sc_results[0]
        sc15, sc15s = sc_results[1]
        sc25, sc25s = sc_results[2]
    else:
        sc8 = sc15 = sc25 = np.full(n_eval, np.float32(last_known_tvt))
        sc8s = sc15s = sc25s = np.zeros(n_eval, np.float32)
    sc_consensus = ((sc8 + sc15 + sc25) / 3.0).astype(np.float32)
    sc_trust = np.float32(np.clip(len(known) / 200.0, 0.0, 0.6))

    # â”€â”€ Affine GR calibration â”€â”€
    tw_at_known  = np.interp(ktvt, tw_tvt, tw_gr).astype(np.float32)
    known_gr_arr = known['GR'].values.astype(np.float32)
    a_cal, b_cal = affine_cal(known_gr_arr, tw_at_known)

    # â”€â”€ Inter-signal consensus â”€â”€
    all_sig = [pf_pred, spatial_tvt, dense_tvt]
    for t in beam_tags:
        arr = beam_preds[t]
        if not np.any(np.isnan(arr)):
            all_sig.append(arr)
    for sc_a in [sc8, sc15, sc25]:
        if not np.any(np.isnan(sc_a)):
            all_sig.append(sc_a)
    valid_sig = [s for s in all_sig if len(s) == n_eval and not np.any(np.isnan(s))]
    signal_std = (np.stack(valid_sig, axis=1).std(1).astype(np.float32)
                  if len(valid_sig) >= 2 else np.zeros(n_eval, np.float32))

    # â”€â”€ GR features â”€â”€
    gr_filled = hw['GR'].astype(float).interpolate(limit_direction='both')
    known_gr_mean = known['GR'].mean()
    if np.isnan(known_gr_mean):
        known_gr_mean = gr_filled.mean()

    gr_full_arr = gr_filled.values.astype(np.float32)
    hgr_env = pd.Series(gr_envelope(gr_full_arr), index=hw.index).iloc[evalz.index].values
    hgr_nrg = pd.Series(gr_energy(gr_full_arr),   index=hw.index).iloc[evalz.index].values

    gr_rolls = {}
    for w_size in [21, 51]:
        roll = gr_filled.rolling(w_size, center=True, min_periods=max(2, w_size // 5))
        gr_rolls[f'gr_roll{w_size}'] = roll.mean().iloc[evalz.index].values.astype(np.float32)
        gr_rolls[f'gr_std{w_size}']  = roll.std().fillna(0).iloc[evalz.index].values.astype(np.float32)

    recent = known.tail(min(200, len(known)))
    slope_all      = robust_slope(known['MD'], known['TVT_input'])
    slope_recent   = robust_slope(recent['MD'], recent['TVT_input'], default=slope_all)
    slope_z_recent = robust_slope(recent['Z'],  recent['TVT_input'])

    known_gr_valid = known[known['GR'].notna()]
    known_tw_rmse  = np.nan
    if len(known_gr_valid) > 5:
        tw_gr_at_k = safe_interp(known_gr_valid['TVT_input'].values, tw_tvt, tw_gr)
        v_mask = np.isfinite(tw_gr_at_k)
        if v_mask.sum() > 5:
            known_tw_rmse = float(np.sqrt(np.mean(
                (known_gr_valid['GR'].values[v_mask] - tw_gr_at_k[v_mask]) ** 2)))

    md_diff = hw['MD'].diff().replace(0, np.nan)
    dz_dmd = (hw['Z'].diff() / md_diff).iloc[evalz.index].values.astype(np.float32)
    dx_dmd = (hw['X'].diff() / md_diff).iloc[evalz.index].values.astype(np.float32)
    dy_dmd = (hw['Y'].diff() / md_diff).iloc[evalz.index].values.astype(np.float32)

    cur = pd.DataFrame({
        'pf_pred':       pf_pred.astype(np.float32),
        'pf_std':        pf_std.astype(np.float32),
        'pf_delta':      (pf_pred - last_known_tvt).astype(np.float32),
        'pf_std_trend':  (pf_std - pf_std[0]).astype(np.float32) if len(pf_std) > 0 else 0.0,
        'pf_std_ratio':  (pf_std / max(pf_std[0], 0.01)).astype(np.float32) if len(pf_std) > 0 else 1.0,
        'spatial_tvt':   spatial_tvt,
        'spatial_delta': (spatial_tvt - last_known_tvt).astype(np.float32),
        'spatial_ancc':  spatial_ancc,
        'spatial_dist':  knn_d,
        'c_well':        np.float32(c_well),
        'dense_tvt':          dense_tvt,
        'dense_delta':        (dense_tvt - last_known_tvt).astype(np.float32),
        'dense_tvt_wls_delta': (dense_tvt_w - last_known_tvt).astype(np.float32),
        'dense_tvt_50_delta':  (dense_tvt_50 - last_known_tvt).astype(np.float32),
        'dense_ancc':         dense_ancc_eval.astype(np.float32),
        'dense_dist':         dense_dist.astype(np.float32),
        'dense_nb_std':       dense_nb_std_eval,
        'dense_known_rmse':   dense_known_rmse,
        'dense_known_bias':   dense_known_bias,
        'dense_known_max_err': dense_known_max_err,
        'dense_known_nb_std': dense_known_nb_std_mean,
        **{f'tvtF_{fn}_d':   (tvt_forms[fn] - last_known_tvt).astype(np.float32) for fn in FORMATIONS},
        **{f'tvtFw_{fn}_d':  (tvt_forms[fn + '_wls'] - last_known_tvt).astype(np.float32) for fn in FORMATIONS},
        **{f'b_{fn}':        tvt_forms[f'b_{fn}'] for fn in FORMATIONS},
        **{f'bw_{fn}':       tvt_forms[f'bw_{fn}'] for fn in FORMATIONS},
        **{f'form_rmse_{fn}': np.float32(form_rmse_vals[fn]) for fn in FORMATIONS},
        'form_mean_d':   form_mean_d,
        'form_std_d':    form_std_d,
        'form_range_d':  form_range_d,
        **{f'beam_{tag}_d': (beam_preds[tag] - np.float32(last_known_tvt)).astype(np.float32)
           for _, _, _, _, tag in BEAMS},
        'beam_mean_d':   beam_mean_d,
        'beam_std_d':    beam_std_d,
        'beam_med_d':    beam_med_d,
        'sc8_d':         (sc8  - np.float32(last_known_tvt)).astype(np.float32),
        'sc8_score':     sc8s,
        'sc15_d':        (sc15 - np.float32(last_known_tvt)).astype(np.float32),
        'sc15_score':    sc15s,
        'sc25_d':        (sc25 - np.float32(last_known_tvt)).astype(np.float32),
        'sc25_score':    sc25s,
        'sc_cons_d':     (sc_consensus - np.float32(last_known_tvt)).astype(np.float32),
        'sc_trust':      np.float32(sc_trust),
        'signal_std':    signal_std,
        'cal_a':         np.float32(a_cal),
        'cal_b':         np.float32(b_cal),
        'pf_vs_spatial':     (pf_pred - spatial_tvt).astype(np.float32),
        'pf_vs_spatial_abs': np.abs(pf_pred - spatial_tvt).astype(np.float32),
        'pf_vs_dense':       (pf_pred - dense_tvt).astype(np.float32),
        'spatial_vs_dense':  (spatial_tvt - dense_tvt).astype(np.float32),
        'pf_vs_beam_cons':   (pf_pred - beam_preds['cons']).astype(np.float32),
        'sc15_vs_beam_cons': (sc15 - beam_preds['cons']).astype(np.float32),
        'md_from_ps':    (evalz['MD'].values - ps_md).astype(np.float32),
        'md_from_ps_sq': ((evalz['MD'].values - ps_md) ** 2 / 1000.0).astype(np.float32),
        'z_from_ps':     (evalz['Z'].values - ps_z).astype(np.float32),
        'x_from_ps':     (evalz['X'].values - ps_x).astype(np.float32),
        'y_from_ps':     (evalz['Y'].values - ps_y).astype(np.float32),
        'xy_dist':       np.sqrt((evalz['X'].values - ps_x) ** 2
                                 + (evalz['Y'].values - ps_y) ** 2).astype(np.float32),
        'eval_frac':     (np.arange(n_eval) / max(n_eval - 1, 1)).astype(np.float32),
        'z':             z_eval,
        'dz_dmd':        dz_dmd,
        'dx_dmd':        dx_dmd,
        'dy_dmd':        dy_dmd,
        'gr':            evalz['GR'].values.astype(np.float32),
        'gr_env':        hgr_env.astype(np.float32),
        'gr_energy':     hgr_nrg.astype(np.float32),
        'last_known_tvt': np.float32(last_known_tvt),
        'known_len':      np.int32(len(known)),
        'eval_len':       np.int32(n_eval),
        'eval_len_ratio': np.float32(n_eval / max(len(known), 1)),
        'slope_all':      np.float32(slope_all),
        'slope_recent':   np.float32(slope_recent),
        'slope_z_recent': np.float32(slope_z_recent),
        'known_tvt_range': np.float32(known['TVT_input'].max() - known['TVT_input'].min()),
        'known_tvt_std':   np.float32(known['TVT_input'].std()),
        'known_gr_mean':   np.float32(known_gr_mean),
        'known_gr_std':    np.float32(known['GR'].std()),
        'known_tw_rmse':   np.float32(known_tw_rmse),
        'tw_tvt_range':    np.float32(tw_tvt.max() - tw_tvt.min()),
        'tw_gr_mean':      np.float32(tw_gr.mean()),
        'tw_gr_std':       np.float32(tw_gr.std()),
    })

    for k_roll, v_roll in gr_rolls.items():
        cur[k_roll] = v_roll

    if not np.isnan(known_gr_mean):
        gr_cumdev = (gr_filled - known_gr_mean).cumsum()
        cur['gr_cumdev'] = gr_cumdev.iloc[evalz.index].values.astype(np.float32)
    else:
        cur['gr_cumdev'] = np.float32(np.nan)

    cur['tw_gr_at_pf']       = safe_interp(pf_pred, tw_tvt, tw_gr).astype(np.float32)
    cur['gr_minus_tw_at_pf'] = (cur['gr'].values - cur['tw_gr_at_pf'].values).astype(np.float32)

    for offset in [-60, 60]:
        tw_gr_off = safe_interp(pf_pred + offset, tw_tvt, tw_gr)
        cur[f'gr_tw_off_{offset}'] = (cur['gr'].values - tw_gr_off).astype(np.float32)

    cur['baseline_slope_recent'] = (last_known_tvt + slope_recent * cur['md_from_ps'].values).astype(np.float32)
    cur['pf_minus_slope']        = (pf_pred - cur['baseline_slope_recent'].values).astype(np.float32)
    cur['spatial_minus_slope']   = (spatial_tvt - cur['baseline_slope_recent'].values).astype(np.float32)
    cur['dense_minus_slope']     = (dense_tvt - cur['baseline_slope_recent'].values).astype(np.float32)

    if is_train:
        cur['target_tvt']      = evalz['TVT'].values.astype(np.float32)
        cur['target_residual'] = (evalz['TVT'].values - last_known_tvt).astype(np.float32)

    return cur

In [ ]:
# â”€â”€ Cal-Zone Augmentation â”€â”€
def build_augmented_features(hw, tw, form_imputer, dense_imputer, wid, tw_tvt, tw_gr,
                              n_splits=N_AUG_SPLITS, min_known=MIN_KNOWN_FOR_AUG):
    known_all = hw[hw['TVT_input'].notna()]
    n_cal = len(known_all)
    if n_cal < min_known + 5:
        return pd.DataFrame()

    split_ks = np.unique(np.linspace(min_known, n_cal - 2, n_splits).astype(int))
    parts = []

    for k in split_ks:
        hw_m = hw.copy()
        mask_start = known_all.index[k]
        hw_m.loc[mask_start:, 'TVT_input'] = np.nan

        known_m = hw_m[hw_m['TVT_input'].notna()]
        evalz_m = hw_m[hw_m['TVT_input'].isna()]
        n_new   = n_cal - k

        if len(evalz_m) == 0 or len(known_m) < 10 or n_new == 0:
            continue

        pf_pred_m, pf_std_m = run_pf_z_velocity(hw_m, tw_tvt, tw_gr)
        ancc_pred_m, ancc_std_m = run_pf_ancc(hw_m, tw_tvt, tw_gr)

        if len(ancc_pred_m) > 0 and not np.any(np.isnan(ancc_pred_m)):
            pf_use = ancc_pred_m.astype(np.float32)
            std_use = ancc_std_m.astype(np.float32)
        elif len(pf_pred_m) > 0:
            pf_use = pf_pred_m.astype(np.float32)
            std_use = pf_std_m.astype(np.float32)
        else:
            continue

        beams_dict = run_all_beams(hw_m, tw)

        gr_filled_m = hw_m['GR'].astype(float).interpolate(limit_direction='both')
        gr_filled_m = gr_filled_m.fillna(float(np.nanmean(tw_gr)))
        kgr_m   = gr_filled_m.loc[known_m.index].values.astype(np.float32)
        hgr_m   = gr_filled_m.iloc[evalz_m.index[0]:].values.astype(np.float32)
        ktvt_m  = known_m['TVT_input'].values.astype(np.float32)
        sc_results = multi_scale_sc(kgr_m, ktvt_m, hgr_m)

        feat = build_features(hw_m, tw, pf_use, std_use, form_imputer, dense_imputer, wid,
                              beams_dict=beams_dict, sc_results=sc_results, is_train=True)
        if len(feat) == 0:
            continue

        feat_new = feat.iloc[:n_new].copy()
        feat_new['aug_split_k'] = np.int32(k)
        parts.append(feat_new)

    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()


def process_single_well_train(hw_path, form_imputer, dense_imputer):
    well_id = Path(hw_path).name.split('__', 1)[0]
    tw_path = TRAIN_DIR / f'{well_id}__typewell.csv'
    try:
        if not tw_path.exists():
            return pd.DataFrame()

        hw = pd.read_csv(hw_path)
        tw = pd.read_csv(tw_path)

        if not {'TVT', 'GR'}.issubset(tw.columns) or len(tw) < 2:
            return pd.DataFrame()

        known = hw[hw['TVT_input'].notna()]
        if len(known) < 10:
            return pd.DataFrame()

        tw_tvt = tw['TVT'].to_numpy(dtype=np.float32)
        tw_gr  = tw['GR'].to_numpy(dtype=np.float32)
        parts  = []

        eval_with_gt = hw[hw['TVT_input'].isna() & hw['TVT'].notna()]
        if len(eval_with_gt) > 0:
            pf_pred, pf_std = run_pf_z_velocity(hw, tw_tvt, tw_gr)
            ancc_pred, ancc_std = run_pf_ancc(hw, tw_tvt, tw_gr)

            if len(ancc_pred) > 0 and not np.any(np.isnan(ancc_pred)):
                pf_use = ancc_pred.astype(np.float32); std_use = ancc_std.astype(np.float32)
            elif len(pf_pred) > 0:
                pf_use = pf_pred.astype(np.float32); std_use = pf_std.astype(np.float32)
            else:
                pf_use = None

            if pf_use is not None:
                beams_dict = run_all_beams(hw, tw)
                gr_filled = hw['GR'].astype(float).interpolate(limit_direction='both')
                gr_filled = gr_filled.fillna(float(np.nanmean(tw_gr)))
                evalz = hw[hw['TVT_input'].isna()]
                kgr  = gr_filled.loc[known.index].values.astype(np.float32)
                hgr  = gr_filled.iloc[evalz.index[0]:].values.astype(np.float32)
                ktvt = known['TVT_input'].values.astype(np.float32)
                sc_results = multi_scale_sc(kgr, ktvt, hgr)

                feat_orig = build_features(
                    hw, tw, pf_use, std_use, form_imputer, dense_imputer, well_id,
                    beams_dict=beams_dict, sc_results=sc_results, is_train=True)
                if len(feat_orig) > 0:
                    feat_orig['aug_split_k'] = np.int32(-1)
                    parts.append(feat_orig)

        feat_aug = build_augmented_features(
            hw, tw, form_imputer, dense_imputer, well_id, tw_tvt, tw_gr)
        if len(feat_aug) > 0:
            parts.append(feat_aug)

        if not parts:
            return pd.DataFrame()

        result = pd.concat(parts, ignore_index=True)
        result['well_id'] = well_id
        return result

    except Exception:
        return pd.DataFrame()


def process_test_well_for_training(hw_path, form_imputer, dense_imputer):
    well_id = Path(hw_path).name.split('__', 1)[0]
    tw_path = TEST_DIR / f'{well_id}__typewell.csv'
    try:
        if not tw_path.exists():
            return pd.DataFrame()
        hw = pd.read_csv(hw_path)
        tw = pd.read_csv(tw_path)
        if not {'TVT', 'GR'}.issubset(tw.columns) or len(tw) < 2:
            return pd.DataFrame()
        known = hw[hw['TVT_input'].notna()]
        if len(known) < MIN_KNOWN_FOR_AUG + 5:
            return pd.DataFrame()
        hw_with_tvt = hw.copy()
        hw_with_tvt['TVT'] = hw_with_tvt['TVT_input']
        tw_tvt = tw['TVT'].to_numpy(dtype=np.float32)
        tw_gr  = tw['GR'].to_numpy(dtype=np.float32)
        feat_aug = build_augmented_features(
            hw_with_tvt, tw, form_imputer, dense_imputer, well_id, tw_tvt, tw_gr)
        if len(feat_aug) > 0:
            feat_aug['well_id'] = well_id
        return feat_aug
    except Exception:
        return pd.DataFrame()

In [ ]:
# â”€â”€ Build spatial imputers from all training wells â”€â”€
t_start = time.time()
hw_files = sorted(glob.glob(str(TRAIN_DIR / '*__horizontal_well.csv')))
train_well_ids = [Path(f).name.split('__', 1)[0] for f in hw_files]

if DEMO:
    hw_files = hw_files[:30]
    print(f'DEMO mode: using {len(hw_files)} wells')

print(f'Building imputers from {len(train_well_ids)} training wells...')

form_imputer  = FormationPlaneKNN(train_well_ids, TRAIN_DIR)
dense_imputer = DenseANCCImputer(train_well_ids, TRAIN_DIR)
print(f'  FormationPlaneKNN: {len(form_imputer.df)} wells | Dense: {len(dense_imputer.ancc)} points')
print(f'  {time.time() - t_start:.1f}s')

In [ ]:
# â”€â”€ Parallel train feature building â”€â”€
print(f'Building train features ({len(hw_files)} wells, N_AUG_SPLITS={N_AUG_SPLITS}, N_JOBS={N_JOBS})...')
t0 = time.time()

results = Parallel(n_jobs=N_JOBS, backend='loky', verbose=5)(
    delayed(process_single_well_train)(hw_path, form_imputer, dense_imputer)
    for hw_path in hw_files
)

parts = [r for r in results if len(r) > 0]
print(f'  Done: {len(parts)} wells produced features ({time.time()-t0:.0f}s)')
del results
gc.collect()

In [ ]:
# â”€â”€ Online training: add test-well cal-zone training samples â”€â”€
test_files = sorted(glob.glob(str(TEST_DIR / '*__horizontal_well.csv')))

if ONLINE_TRAINING:
    print(f'Online training: processing {len(test_files)} test wells...')
    t0 = time.time()
    test_train_results = Parallel(n_jobs=min(N_JOBS, len(test_files)), backend='loky', verbose=1)(
        delayed(process_test_well_for_training)(hw_path, form_imputer, dense_imputer)
        for hw_path in test_files
    )
    test_parts = [r for r in test_train_results if len(r) > 0]
    print(f'  Test wells with training samples: {len(test_parts)} ({time.time()-t0:.1f}s)')
    del test_train_results
    parts.extend(test_parts)
    gc.collect()

train_df = pd.concat(parts, ignore_index=True)
del parts
gc.collect()

orig_count  = (train_df['aug_split_k'] == -1).sum()
aug_count   = (train_df['aug_split_k'] >= 0).sum()
print(f'\ntrain_df: {train_df.shape} | wells: {train_df["well_id"].nunique()}')
print(f'  Original hidden-zone rows: {orig_count:,}')
print(f'  Augmented cal-zone rows:   {aug_count:,} (+{aug_count/max(orig_count,1)*100:.0f}%)')

In [ ]:
# â”€â”€ Train LightGBM â”€â”€
EXCLUDE = {'well_id', 'target_tvt', 'target_residual', 'aug_split_k'}
FEATURE_COLS = [c for c in train_df.columns if c not in EXCLUDE]
print(f'Features ({len(FEATURE_COLS)}): {FEATURE_COLS[:10]}...')

X_train = train_df[FEATURE_COLS].astype(np.float32)
y_train = train_df['target_residual'].astype(np.float32)
del train_df
gc.collect()

print(f'\nTraining LightGBM on {len(X_train):,} rows...')
t0 = time.time()
model = LGBMRegressor(**LGBM_PARAMS)
model.fit(X_train, y_train)
print(f'Training done in {time.time()-t0:.1f}s')

del X_train, y_train
gc.collect()

In [ ]:
# â”€â”€ Build test features & predict â”€â”€
np.random.seed(RANDOM_STATE)
t0 = time.time()

test_files = sorted(glob.glob(str(TEST_DIR / '*__horizontal_well.csv')))
print(f'Test wells: {len(test_files)}')

rows = []
fallback_wells = []

for i, hw_path in enumerate(test_files, 1):
    well_id = Path(hw_path).name.split('__', 1)[0]
    tw_path = TEST_DIR / f'{well_id}__typewell.csv'
    h = pd.read_csv(hw_path)
    evalz = h[h['TVT_input'].isna()]
    if len(evalz) == 0:
        continue

    known = h[h['TVT_input'].notna()]
    if len(known) < 10 or not tw_path.exists():
        fallback = float(known['TVT_input'].iloc[-1]) if len(known) > 0 else 0.0
        fallback_wells.append(well_id)
        for _, row in evalz.iterrows():
            rows.append({'id': f'{well_id}_{row.name}', 'tvt': fallback})
        continue

    tw = pd.read_csv(tw_path)
    if not {'TVT', 'GR'}.issubset(tw.columns) or len(tw) < 2:
        fallback = float(known['TVT_input'].iloc[-1])
        fallback_wells.append(well_id)
        for _, row in evalz.iterrows():
            rows.append({'id': f'{well_id}_{row.name}', 'tvt': fallback})
        continue

    tw_tvt = tw['TVT'].to_numpy(dtype=np.float32)
    tw_gr  = tw['GR'].to_numpy(dtype=np.float32)

    pf_pred, pf_std = run_pf_z_velocity(h, tw_tvt, tw_gr)
    ancc_pred, ancc_std = run_pf_ancc(h, tw_tvt, tw_gr)

    if len(ancc_pred) > 0 and not np.any(np.isnan(ancc_pred)):
        pf_use = ancc_pred.astype(np.float32); std_use = ancc_std.astype(np.float32)
    elif len(pf_pred) > 0:
        pf_use = pf_pred.astype(np.float32); std_use = pf_std.astype(np.float32)
    else:
        fallback = float(known['TVT_input'].iloc[-1])
        fallback_wells.append(well_id)
        for _, row in evalz.iterrows():
            rows.append({'id': f'{well_id}_{row.name}', 'tvt': fallback})
        continue

    beams_dict = run_all_beams(h, tw)

    gr_filled = h['GR'].astype(float).interpolate(limit_direction='both')
    gr_filled = gr_filled.fillna(float(np.nanmean(tw_gr)))
    kgr  = gr_filled.loc[known.index].values.astype(np.float32)
    hgr  = gr_filled.iloc[evalz.index[0]:].values.astype(np.float32)
    ktvt = known['TVT_input'].values.astype(np.float32)
    sc_results = multi_scale_sc(kgr, ktvt, hgr)

    feat = build_features(h, tw, pf_use, std_use, form_imputer, dense_imputer, well_id,
                          beams_dict=beams_dict, sc_results=sc_results, is_train=False)

    if len(feat) > 0:
        X_test = feat[FEATURE_COLS].astype(np.float32)
        pred_delta = model.predict(X_test).astype(np.float32)
        lkt = feat['last_known_tvt'].iloc[0]
        pred_tvt = lkt + pred_delta
        for j, (_, row) in enumerate(evalz.iterrows()):
            rows.append({'id': f'{well_id}_{row.name}', 'tvt': float(pred_tvt[j])})
    else:
        fallback = float(known['TVT_input'].iloc[-1])
        fallback_wells.append(well_id)
        for _, row in evalz.iterrows():
            rows.append({'id': f'{well_id}_{row.name}', 'tvt': fallback})

    if i % 50 == 0:
        print(f'  {i}/{len(test_files)} wells ({time.time()-t0:.0f}s)')

print(f'\nDone in {time.time()-t0:.0f}s')
if fallback_wells:
    print(f'Fallback wells: {len(fallback_wells)}')

In [ ]:
sub = pd.DataFrame(rows)
sub.to_csv(OUT_DIR / 'submission.csv', index=False)
print(f'Submission rows: {len(sub)}')
print(sub.head())

sample_path = DATA_DIR / 'sample_submission.csv'
if sample_path.exists():
    sample = pd.read_csv(sample_path)
    print(f'\nSample rows: {len(sample)} | Our rows: {len(sub)}')
    print(f'IDs match: {set(sub["id"]) == set(sample["id"])}')